# Qwen2.5 Coder C++ Review QLoRA Training on Kaggle

Run this notebook from top to bottom. It copies your read-only Kaggle project files into `/kaggle/working`, configures the dataset path, installs dependencies with `uv`, and starts training. Re-running the training cell resumes automatically from the latest checkpoint.

In [ ]:
#loading dataset to checkpoint

import os
import shutil
from IPython.display import display, Markdown

# --- Configuration ---
SOURCE_DIR = '/kaggle/input/datasets/saffiullah892/qwen2-5-01'
DESTINATION_DIR = '/kaggle/working'

display(Markdown("## 1️⃣ Creating Destination Folder"))

# Create the destination folder if it doesn't exist
os.makedirs(DESTINATION_DIR, exist_ok=True)
print(f"Destination folder created/verified: {DESTINATION_DIR}")

# --- Check Source Contents ---
display(Markdown("## 2️⃣ Verifying Source and Copying Files"))

if not os.path.exists(SOURCE_DIR):
    print(f"❌ ERROR: Source directory not found at {SOURCE_DIR}")
    print("Please ensure you have added 'my-checkpoints-dataset-v2' as a data source to this notebook.")
else:
    # Use shutil.copytree to copy the entire folder contents
    try:
        # We use os.listdir to check for the zip file and copy only its contents
        # NOTE: If your 'my-checkpoints-dataset-v2' contains a ZIP file (e.g., all_checkpoints.zip),
        # this copy command will just copy the ZIP file. You may need to UNZIP it after copying.

        # If you are sure there is only one file/folder in the source and you want everything copied:
        for item in os.listdir(SOURCE_DIR):
            source_item = os.path.join(SOURCE_DIR, item)
            destination_item = os.path.join(DESTINATION_DIR, item)
            
            if os.path.isdir(source_item):
                # Copy entire subdirectory
                shutil.copytree(source_item, destination_item, dirs_exist_ok=True)
            else:
                # Copy individual file
                shutil.copy2(source_item, destination_item)
        
        print(f"✅ Successfully copied all contents from {SOURCE_DIR} to {DESTINATION_DIR}")
        
        # Verify the contents of the new folder
        print("\nContents of the new folder:")
        !ls -l {DESTINATION_DIR}

    except Exception as e:
        print(f"❌ An error occurred during copy: {e}")

In [ ]:
!nvidia-smi
!python --version
!python -m pip install --upgrade uv

## Copy Project to Writable Storage

Kaggle mounts `/kaggle/input` as read-only. Training needs to write configs, checkpoints, adapters, `.pth` files, and ONNX exports, so the project is copied to `/kaggle/working/project-files`.

In [ ]:
from pathlib import Path
import shutil

PROJECT_INPUT = Path('/kaggle/input/datasets/saffiullah892/project-files')
WORK_REPO = Path('/kaggle/working/project-files')
DATASET_FILE = Path('/kaggle/input/datasets/saffiullah892/clean-datset01/merged_cleaned.jsonl')

assert PROJECT_INPUT.exists(), f'Project folder not found: {PROJECT_INPUT}'
assert DATASET_FILE.exists(), f'Training dataset not found: {DATASET_FILE}'

shutil.copytree(PROJECT_INPUT, WORK_REPO, dirs_exist_ok=True)
%cd /kaggle/working/project-files

assert Path('pyproject.toml').exists(), 'pyproject.toml missing after copy'
print('Project copied to:', WORK_REPO)
print('Training dataset:', DATASET_FILE)

## Patch Adapter-Only Continue Support

This notebook can be uploaded alone. This cell patches the copied project files in `/kaggle/working/project-files` before `uv sync`, so old Kaggle project datasets still get the adapter-only continuation fix.

In [ ]:
from pathlib import Path

config_py = Path('src/qwen_cpp_review/config.py')
trainer_py = Path('src/qwen_cpp_review/trainer.py')
assert config_py.exists(), f'Missing {config_py}'
assert trainer_py.exists(), f'Missing {trainer_py}'

config_text = config_py.read_text()
if 'initial_adapter_path' not in config_text:
    config_text = config_text.replace(
        '    resume_from_checkpoint: str | None = None\n',
        '    resume_from_checkpoint: str | None = None\n    initial_adapter_path: str | None = None\n',
    )
    config_py.write_text(config_text)
    print('Patched config.py: added initial_adapter_path')
else:
    print('config.py already supports initial_adapter_path')

trainer_text = trainer_py.read_text()
if 'from peft import PeftModel' not in trainer_text:
    trainer_text = trainer_text.replace('import torch\n', 'import torch\nfrom peft import PeftModel\n')

old = '    peft_config = create_lora_config(config.lora)\n'
new = (
    '    peft_config = None if config.training.initial_adapter_path else create_lora_config(config.lora)\n'
    '    if config.training.initial_adapter_path:\n'
    '        LOGGER.info("loading initial adapter weights: %s", config.training.initial_adapter_path)\n'
    '        model = PeftModel.from_pretrained(model, config.training.initial_adapter_path, is_trainable=True)\n'
)
if old in trainer_text and 'loading initial adapter weights' not in trainer_text:
    trainer_text = trainer_text.replace(old, new)

old = '    resume = config.training.resume_from_checkpoint or find_latest_checkpoint(output_dir)\n'
new = (
    '    resume = None\n'
    '    if not config.training.initial_adapter_path:\n'
    '        resume = config.training.resume_from_checkpoint or find_latest_checkpoint(output_dir)\n'
)
if old in trainer_text:
    trainer_text = trainer_text.replace(old, new)

old = '    if resume:\n        LOGGER.info("resuming from checkpoint: %s", resume)\n    trainer.train(resume_from_checkpoint=resume)\n'
new = (
    '    if resume:\n'
    '        LOGGER.info("resuming from checkpoint: %s", resume)\n'
    '    elif config.training.initial_adapter_path:\n'
    '        LOGGER.info("continuing from adapter weights without optimizer-state resume")\n'
    '    trainer.train(resume_from_checkpoint=resume)\n'
)
if old in trainer_text:
    trainer_text = trainer_text.replace(old, new)

trainer_py.write_text(trainer_text)
assert 'initial_adapter_path' in config_py.read_text()
assert 'loading initial adapter weights' in trainer_py.read_text()
assert 'continuing from adapter weights without optimizer-state resume' in trainer_py.read_text()
print('Adapter-only continue patch is active in working project files')


## Install Dependencies with uv

In [ ]:
!mkdir -p /kaggle/temp/uv-cache /kaggle/temp/project-venv /kaggle/temp/hf-cache /kaggle/temp/hf-datasets
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache UV_LINK_MODE=copy uv sync --extra gpu --extra export --extra dev
!du -sh /kaggle/temp/project-venv /kaggle/temp/uv-cache /kaggle/working/project-files || true

## Configure Training

This cell writes your Kaggle dataset path into `configs/train_qlora.yaml` and sends all outputs to `/kaggle/working/outputs`, which is writable and downloadable.

In [ ]:
import yaml
from pathlib import Path

DATASET_FILE = '/kaggle/input/datasets/saffiullah892/clean-datset01/merged_cleaned.jsonl'
OUTPUT_DIR = '/kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'

config_path = Path('configs/train_qlora.yaml')
config = yaml.safe_load(config_path.read_text())

config['data']['data_files'] = [DATASET_FILE]
config['data']['cache_dir'] = '/kaggle/temp/hf-datasets'
config['data']['identifier_augmentation'] = True
config['data']['identifier_augmentation_copies'] = 1
config['training']['output_dir'] = OUTPUT_DIR
config['training']['resume_from_checkpoint'] = None
config['training']['initial_adapter_path'] = '/kaggle/input/datasets/saffiullah892/qwen2-5-01/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/checkpoint-750'
config['training']['logging_steps'] = 1
config['training']['packing'] = False
config['training']['gradient_checkpointing_use_reentrant'] = False
config['training']['ddp_find_unused_parameters'] = False
config['training']['save_total_limit'] = 2
config['training']['save_steps'] = 250
config['training']['eval_steps'] = 250

config_path.write_text(yaml.safe_dump(config, sort_keys=False))

print('Configured dataset:', config['data']['data_files'])
print('Configured output:', config['training']['output_dir'])
print('Adapter-only continue from:', config['training']['initial_adapter_path'])


## Optional Sanity Check

This checks that the dataset file is readable and shows the first row keys.

In [ ]:
import json
from pathlib import Path

dataset_path = Path('/kaggle/input/datasets/saffiullah892/clean-datset01/merged_cleaned.jsonl')
with dataset_path.open() as handle:
    first = json.loads(handle.readline())

print('Dataset size GB:', round(dataset_path.stat().st_size / 1024**3, 3))
print('First row keys:', sorted(first.keys()))

## Check Resume Point

This shows exactly which checkpoint exists before training starts. `train.py` resumes from the latest one automatically.

In [ ]:
from pathlib import Path
import json

OUTPUT_DIR = Path('/kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora')
checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
print('Output dir:', OUTPUT_DIR)
print('Checkpoints found:', [p.name for p in checkpoints])
if checkpoints:
    latest = checkpoints[-1]
    print('Will resume from:', latest)
    state_file = latest / 'trainer_state.json'
    if state_file.exists():
        state = json.loads(state_file.read_text())
        print('global_step:', state.get('global_step'))
        print('epoch:', state.get('epoch'))
        print('best_model_checkpoint:', state.get('best_model_checkpoint'))
else:
    print('No checkpoint found. Training will start from step 0.')


## Optional: Adapter-Only Continue

Use this if exact Trainer resume fails with a bitsandbytes optimizer-state error. It continues from learned LoRA weights but starts a fresh optimizer/scheduler.

In [ ]:
# Adapter-only continue is enabled by default to avoid bitsandbytes optimizer-state resume errors.
ADAPTER_ONLY_CONTINUE = True
INPUT_CHECKPOINT = '/kaggle/input/datasets/saffiullah892/qwen2-5-01/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/checkpoint-750'

import yaml
from pathlib import Path

config_path = Path('configs/train_qlora.yaml')
config = yaml.safe_load(config_path.read_text())
if ADAPTER_ONLY_CONTINUE:
    config['training']['initial_adapter_path'] = INPUT_CHECKPOINT
    config['training']['resume_from_checkpoint'] = None
    print('Adapter-only continue from:', INPUT_CHECKPOINT)
else:
    config['training']['initial_adapter_path'] = None
    print('Exact checkpoint resume mode')
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('Config initial_adapter_path:', config['training'].get('initial_adapter_path'))
print('Config resume_from_checkpoint:', config['training'].get('resume_from_checkpoint'))


## Verify Updated Resume Code

This fails early if Kaggle is still using old project files that do not support adapter-only continuation.

In [ ]:
!grep -n 'initial_adapter_path' src/qwen_cpp_review/config.py src/qwen_cpp_review/trainer.py
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python - <<'PY'
from qwen_cpp_review.config import TrainingConfig
import inspect
import qwen_cpp_review.trainer as trainer
assert hasattr(TrainingConfig(), 'initial_adapter_path'), 'Old config.py loaded: missing initial_adapter_path'
source = inspect.getsource(trainer.train)
assert 'initial_adapter_path' in source, 'Old trainer.py loaded: missing adapter-only continue logic'
print('Updated adapter-only training code is active')
PY


## Start or Resume Training

This cell detects the number of GPUs and chooses single-GPU or multi-GPU Accelerate launch. If checkpoints already exist in `OUTPUT_DIR`, `train.py` resumes from the newest `checkpoint-*` automatically.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
export HF_HOME=/kaggle/temp/hf-cache
export HF_DATASETS_CACHE=/kaggle/temp/hf-datasets
export TRANSFORMERS_CACHE=/kaggle/temp/hf-cache
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export ACCELERATE_LOG_LEVEL=info
export TRANSFORMERS_VERBOSITY=info
export NO_COLOR=1
LOG_FILE=/kaggle/working/train.log
echo "Training log: ${LOG_FILE}"
echo "Started at: $(date)" | tee -a "${LOG_FILE}"

NUM_GPUS=$(uv run python - <<'PY'
import torch
print(torch.cuda.device_count())
PY
)

echo "Detected GPUs: ${NUM_GPUS}" | tee -a "${LOG_FILE}"
MODE_INFO=$(uv run python - <<'PY'
import yaml
cfg = yaml.safe_load(open("configs/train_qlora.yaml"))
tr = cfg.get("training", {})
adapter = tr.get("initial_adapter_path")
resume = tr.get("resume_from_checkpoint")
print(f"Config initial_adapter_path: {adapter}")
print(f"Config resume_from_checkpoint: {resume}")
if adapter:
    print(f"Adapter-only continue from: {adapter}")
else:
    print("Exact Trainer resume mode")
PY
)
echo "${MODE_INFO}" | tee -a "${LOG_FILE}"

if ! echo "${MODE_INFO}" | grep -q "Adapter-only continue from:"; then
  LATEST_CKPT=$(find /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora -maxdepth 1 -type d -name "checkpoint-*" 2>/dev/null | sort -V | tail -1 || true)
  if [ -n "${LATEST_CKPT}" ]; then
    echo "Will exact-resume from: ${LATEST_CKPT}" | tee -a "${LOG_FILE}"
  else
    echo "No checkpoint found. Training starts from step 0." | tee -a "${LOG_FILE}"
  fi
fi

if [ "${NUM_GPUS}" -gt 1 ]; then
  uv run accelerate launch \
    --config_file configs/accelerate_multi_gpu.yaml \
    --num_processes "${NUM_GPUS}" \
    train.py --config configs/train_qlora.yaml 2>&1 | tee -a "${LOG_FILE}"
else
  uv run accelerate launch \
    --config_file configs/accelerate_single_gpu.yaml \
    train.py --config configs/train_qlora.yaml 2>&1 | tee -a "${LOG_FILE}"
fi


## View Training Log

Run this after training finishes, or from the Kaggle console while training is running.

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv tail -n 80 /kaggle/working/train.log || true

## Check Saved Training Outputs

Expected outputs include `best_adapter/`, `best_adapter.pth`, `last_adapter/`, `last_adapter.pth`, `final_adapter/`, and `final_adapter.pth`.

In [ ]:
!ls -lah /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora || true
!find /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora -maxdepth 2 -type f \( -name '*.pth' -o -name 'adapter_model.safetensors' -o -name 'trainer_state.json' \) -print || true

## Evaluate Best Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python evaluate.py \
  --config configs/train_qlora.yaml \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter

## Test Checkpoint 750

This uses the default Kaggle checkpoint path inside `scripts/test_model.py`, tests easy/medium/hard C++ examples, then includes renamed-variable checks.

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python scripts/test_model.py
!head -2 /kaggle/working/outputs/model_test_predictions.jsonl || true

## Merge Best LoRA Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python merge_lora.py \
  --base-model Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter \
  --output-dir /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged

## Export Merged Model to ONNX

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python export_onnx.py \
  --model /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged \
  --output /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review.onnx

## Final Files to Download

Download from `/kaggle/working/outputs` after training finishes.

In [ ]:
!find /kaggle/working/outputs -maxdepth 3 -type f \( -name '*.pth' -o -name '*.onnx' -o -name 'adapter_model.safetensors' -o -name 'model.safetensors' -o -name 'training_config.yaml' \) -print || true